# Upload Qwen2-VL-2B OMR LoRA to Hugging Face

Upload your finished `checkpoint-9000` LoRA adapter to an **existing** Hugging Face model repository.

## Before you run

1. Create a Hugging Face **Write** token at https://huggingface.co/settings/tokens
2. Default target repo is `NathanZ721/omr-monophonic` (change `HF_REPO_ID` in cell 1 only if needed)
3. Put these files on Google Drive at `MyDrive/omr/qwen2-vl-2b-omr-lora/`:
   - `adapter_model.safetensors` (~675 MB)
   - `adapter_config.json`
   - `README.md` (recommended; included in this repo)
4. Do **not** upload training-only files: `optimizer.pt`, `scheduler.pt`, `trainer_state.json`, `training_args.bin`, `rng_state.pth`

CPU runtime is enough for upload. GPU is only needed if you run the optional smoke-test cell at the end.

In [ ]:
# @title 1) Mount Drive and configure upload
from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

HF_TOKEN = ""  # @param {type:"string"}
HF_REPO_ID = "NathanZ721/omr-monophonic"  # @param {type:"string"}

DRIVE_ADAPTER_DIR = Path("/content/drive/MyDrive/omr/qwen2-vl-2b-omr-lora")
LOCAL_UPLOAD_DIR = Path("/content/hf_upload")

BASE_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
CHECKPOINT_LABEL = "checkpoint-9000"
TRAINING_EPOCHS = 2.07

REQUIRED_FILES = [
    "adapter_model.safetensors",
    "adapter_config.json",
]

if not HF_TOKEN.strip():
    raise ValueError("Set HF_TOKEN to a Hugging Face token with Write permission.")
if "/" not in HF_REPO_ID or HF_REPO_ID.startswith("/"):
    raise ValueError("HF_REPO_ID must look like username/repo-name")
if not DRIVE_ADAPTER_DIR.is_dir():
    raise FileNotFoundError(
        f"Adapter folder not found: {DRIVE_ADAPTER_DIR}\n"
        "Upload adapter_model.safetensors and adapter_config.json to that Drive path."
    )

missing = [name for name in REQUIRED_FILES if not (DRIVE_ADAPTER_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(f"Missing required files in {DRIVE_ADAPTER_DIR}: {missing}")

LOCAL_UPLOAD_DIR.mkdir(parents=True, exist_ok=True)
print(f"Drive adapter dir: {DRIVE_ADAPTER_DIR}")
print(f"Target repo:       {HF_REPO_ID}")

In [ ]:
# @title 2) Install huggingface_hub
import subprocess
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub"])

In [ ]:
# @title 3) Stage upload files locally
import shutil

from pathlib import Path

LOCAL_UPLOAD_DIR = Path("/content/hf_upload")
DRIVE_ADAPTER_DIR = Path("/content/drive/MyDrive/omr/qwen2-vl-2b-omr-lora")
REQUIRED_FILES = [
    "adapter_model.safetensors",
    "adapter_config.json",
]

if LOCAL_UPLOAD_DIR.exists():
    shutil.rmtree(LOCAL_UPLOAD_DIR)
LOCAL_UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

for name in REQUIRED_FILES:
    shutil.copy2(DRIVE_ADAPTER_DIR / name, LOCAL_UPLOAD_DIR / name)

readme_src = DRIVE_ADAPTER_DIR / "README.md"
if readme_src.is_file():
    shutil.copy2(readme_src, LOCAL_UPLOAD_DIR / "README.md")
    print("Copied README.md from Drive.")
else:
    print("No README.md on Drive; the next cell will generate one.")

for path in sorted(LOCAL_UPLOAD_DIR.iterdir()):
    print(f"{path.name:28s} {path.stat().st_size / (1024 ** 2):8.1f} MB")

In [ ]:
# @title 4) Write model card README (if needed)
import json
from pathlib import Path

LOCAL_UPLOAD_DIR = Path("/content/hf_upload")
HF_REPO_ID = "NathanZ721/omr-monophonic"  # @param {type:"string"}
HF_LICENSE = "apache-2.0"  # @param ["apache-2.0", "mit", "cc-by-4.0", "other"]
BASE_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"
CHECKPOINT_LABEL = "checkpoint-9000"
TRAINING_EPOCHS = 2.07
OVERWRITE_README = False  # @param {type:"boolean"}

readme_path = LOCAL_UPLOAD_DIR / "README.md"
if readme_path.is_file() and not OVERWRITE_README:
    print(f"Keeping existing README at {readme_path}")
else:
    with open(LOCAL_UPLOAD_DIR / "adapter_config.json", encoding="utf-8") as f:
        adapter_cfg = json.load(f)

    lora_r = adapter_cfg.get("r", "?")
    lora_alpha = adapter_cfg.get("lora_alpha", "?")
    lora_dropout = adapter_cfg.get("lora_dropout", "?")

    readme = f"""---
base_model: {BASE_MODEL_ID}
library_name: peft
pipeline_tag: image-text-to-text
license: {HF_LICENSE}
tags:
- optical-music-recognition
- omr
- primus
- lora
- peft
- qwen2-vl
- base_model:adapter:{BASE_MODEL_ID}
---

# {HF_REPO_ID.split('/')[-1]}

LoRA adapter for monophonic optical music recognition (OMR). Given a sheet-music image, the model returns PrIMuS semantic tokens as a single space-separated string.

## Model details

- **Base model:** `{BASE_MODEL_ID}`
- **Method:** LoRA (PEFT)
- **Checkpoint:** `{CHECKPOINT_LABEL}` ({TRAINING_EPOCHS} epochs)
- **LoRA rank / alpha / dropout:** {lora_r} / {lora_alpha} / {lora_dropout}
- **Output format:** PrIMuS semantic tokens

## Evaluation (100-image golden set)

Scoring uses token-level edit distance with canonicalized key signatures and common/cut-time aliases (`4/4` ↔ `C`, `2/2` ↔ `C/`). At most one trailing barline on either side is forgiven.

| Metric | Overall |
| --- | ---: |
| Token accuracy | 97.91% |
| Exact match | 70% |

## Usage

```python
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel
from PIL import Image

base_id = "{BASE_MODEL_ID}"
adapter_id = "{HF_REPO_ID}"

processor = AutoProcessor.from_pretrained(base_id)
model = Qwen2VLForConditionalGeneration.from_pretrained(base_id, device_map="auto")
model = PeftModel.from_pretrained(model, adapter_id)
model.eval()

image = Image.open("score.png").convert("RGB")
messages = [
    {{
        "role": "user",
        "content": [
            {{"type": "image", "image": image}},
            {{
                "type": "text",
                "text": (
                    "Transcribe this monophonic sheet-music image into PrIMuS semantic format. "
                    "Return only one space-separated token string with no explanation or markdown."
                ),
            }},
        ],
    }}
]

text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = processor(text=[text], images=[image], return_tensors="pt").to(model.device)
output_ids = model.generate(**inputs, max_new_tokens=512)
print(processor.batch_decode(output_ids, skip_special_tokens=True)[0])
```

## Limitations

- Trained for **monophonic** excerpts only.
- Best on PrIMuS-style semantic token output; not a general music understanding model.
- Performance is lower on manually captured handwritten examples than on synthetic data.

## Training data

Fine-tuned on a consolidated OMR dataset (~139k image / PrIMuS semantic pairs) with mixed handwritten, clean digital, and camera-distorted styles.
"""
    readme_path.write_text(readme, encoding="utf-8")
    print(f"Wrote model card to {readme_path}")

if HF_REPO_ID and HF_REPO_ID != "NathanZ721/omr-monophonic":
    text = readme_path.read_text(encoding="utf-8")
    patched = text.replace("NathanZ721/omr-monophonic", HF_REPO_ID)
    readme_path.write_text(patched, encoding="utf-8")
    print(f"Patched README adapter id to {HF_REPO_ID}")

In [ ]:
# @title 5) Verify existing repo and upload
from huggingface_hub import HfApi, upload_folder
from huggingface_hub.utils import RepositoryNotFoundError

HF_TOKEN = ""  # @param {type:"string"}
HF_REPO_ID = "NathanZ721/omr-monophonic"  # @param {type:"string"}
LOCAL_UPLOAD_DIR = "/content/hf_upload"
CHECKPOINT_LABEL = "checkpoint-9000"  # @param {type:"string"}

api = HfApi(token=HF_TOKEN)
user = api.whoami()
print(f"Authenticated as: {user.get('name', user)}")

try:
    repo_info = api.repo_info(repo_id=HF_REPO_ID, repo_type="model", token=HF_TOKEN)
except RepositoryNotFoundError as exc:
    raise FileNotFoundError(
        f"Hugging Face model repo not found: {HF_REPO_ID}\n"
        "Create the repo on huggingface.co first, or fix HF_REPO_ID in cell 1."
    ) from exc

print(f"Found existing repo: https://huggingface.co/{repo_info.id}")

commit_info = upload_folder(
    folder_path=LOCAL_UPLOAD_DIR,
    repo_id=HF_REPO_ID,
    repo_type="model",
    token=HF_TOKEN,
    commit_message=f"Upload {CHECKPOINT_LABEL} LoRA adapter for OMR",
)

print(f"Uploaded to https://huggingface.co/{HF_REPO_ID}")
print(f"Commit: {commit_info.commit_url}")

In [ ]:
# @title 6) Optional smoke test (GPU runtime recommended)
# If this cell previously failed on torchao, use Runtime -> Restart session,
# then rerun only this cell.
RUN_SMOKE_TEST = False  # @param {type:"boolean"}
HF_REPO_ID = "NathanZ721/omr-monophonic"  # @param {type:"string"}
BASE_MODEL_ID = "Qwen/Qwen2-VL-2B-Instruct"  # @param {type:"string"}

if not RUN_SMOKE_TEST:
    print("Skipped. Set RUN_SMOKE_TEST=True to verify the uploaded adapter loads.")
else:
    import importlib.machinery
    import importlib.metadata
    import subprocess
    import sys

    if "peft" in sys.modules:
        raise RuntimeError(
            "peft was already imported in this session. Use Runtime -> Restart session, "
            "then rerun only this cell."
        )

    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-U",
            "transformers>=4.49.0,<5.0.0",
            "accelerate",
            "peft",
            "qwen-vl-utils",
            "safetensors",
        ]
    )
    # Colab ships an old torchao that breaks PEFT import checks. BF16 inference does not need it.
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"])

    try:
        torchao_version = importlib.metadata.version("torchao")
    except importlib.metadata.PackageNotFoundError:
        torchao_version = None

    if torchao_version is not None or importlib.machinery.PathFinder.find_spec("torchao") is not None:
        raise RuntimeError(
            "torchao is still present after uninstall. Restart the runtime, then rerun this cell."
        )

    from peft import PeftModel
    from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

    processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
    model = Qwen2VLForConditionalGeneration.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype="auto",
        device_map="auto",
    )
    model = PeftModel.from_pretrained(model, HF_REPO_ID)
    model.eval()
    print(f"Loaded base={BASE_MODEL_ID} + adapter={HF_REPO_ID}")